# ViT-B16 Fine-tune + Attention Rollout Heatmaps

**Run cells 1 → 2 → 3 → 4 → 5 → 6 → 7 in order on a fresh T4 runtime.**

What this notebook does:
1. Fine-tunes `vit_base_patch16_224` (ImageNet pretrained) on IDRiD training set
2. Saves best checkpoint as `vit_b16_best.pth` (picked by best val quadratic kappa)
3. Runs attention rollout on all 27 frozen test images
4. Saves 27 `.npy` heatmaps + 27 overlay `.png` files

Hyperparameters match `week1_training.ipynb` exactly:
- Optimizer: Adam, lr=1e-4, weight_decay=1e-4
- Scheduler: CosineAnnealingLR
- Loss: CrossEntropyLoss with class weights
- Augmentation: same train_transform as EfficientNet/ResNet runs
- Epochs: 30 (ViT-B16 at 224×224 fits in T4 memory at batch=16)


In [ ]:
# Cell 1: Mount Drive + Clone/Pull Repo
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys, importlib

REPO_URL = 'https://github.com/rao-shreyashree/diabetic-retinopathy-xai.git'
REPO_DIR = '/content/diabetic-retinopathy-xai'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

for folder in ['datasets', 'xai', 'utils']:
    init = os.path.join(REPO_DIR, folder, '__init__.py')
    if not os.path.exists(init):
        open(init, 'w').close()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
print('Repo ready')


In [ ]:
# Cell 2: Install Dependencies
import os
os.system('pip install timm scikit-learn -q')
print('Done')


In [ ]:
# Cell 3: Imports + Paths
import os, sys, json, time, glob
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import cohen_kappa_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import matplotlib.pyplot as plt
import matplotlib.cm as cm

REPO_DIR = '/content/diabetic-retinopathy-xai'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Paths — matches week1_training.ipynb exactly ──────────────────────────────
# NOTE: PROJECT_ROOT has a space in it ("diabetic retinopathy") — this is correct,
# it matches the actual Drive folder name used in week1_training.ipynb.
PROJECT_ROOT = '/content/drive/MyDrive/Projects/diabetic retinopathy/diabetic-retinopathy-xai'
DATA_ROOT    = f'{PROJECT_ROOT}/data/IDRiD'
TRAIN_IMG    = f'{DATA_ROOT}/grading/images/train'
TEST_IMG     = f'{DATA_ROOT}/grading/images/test'
TRAIN_CSV    = f'{DATA_ROOT}/grading/labels/train.csv'
TEST_CSV     = f'{DATA_ROOT}/grading/labels/test.csv'
CKPT_DIR     = f'{PROJECT_ROOT}/results/checkpoints'
HEATMAP_DIR  = f'{PROJECT_ROOT}/results/heatmaps/vit_b16/attention_rollout'
OVERLAY_DIR  = f'{PROJECT_ROOT}/results/overlays/vit_b16'

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)

CKPT_PATH = os.path.join(CKPT_DIR, 'vit_b16_best.pth')

# ── Hyperparameters — same as week1_training.ipynb ───────────────────────────
IMG_SIZE     = 224   # ViT-B16 native resolution
BATCH_SIZE   = 16    # fits T4 at 224×224
NUM_CLASSES  = 5
NUM_EPOCHS   = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42

# Frozen test IDs
with open(os.path.join(REPO_DIR, 'test_image_ids.json')) as f:
    TEST_IDS = json.load(f)

print(f'Train img dir exists: {os.path.exists(TRAIN_IMG)}')
print(f'Train CSV exists:     {os.path.exists(TRAIN_CSV)}')
print(f'Test img dir exists:  {os.path.exists(TEST_IMG)}')
print(f'{len(TEST_IDS)} frozen test IDs: {TEST_IDS[0]} to {TEST_IDS[-1]}')


In [ ]:
# Cell 4: Dataset + Dataloaders

# ── Transforms (identical to week1_training.ipynb) ───────────────────────────
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class IDRiDDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['Retinopathy grade'])
        img   = Image.open(os.path.join(self.img_dir,
                           row['Image name'] + '.jpg')).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Load CSVs
train_df = pd.read_csv(TRAIN_CSV, usecols=['Image name', 'Retinopathy grade'])
test_df  = pd.read_csv(TEST_CSV,  usecols=['Image name', 'Retinopathy grade'])

# Exclude frozen test IDs from training (same logic as week1_training.ipynb)
frozen_nums = {int(x.replace('IDRiD_', '')) for x in TEST_IDS}
def img_num(name):
    return int(name.replace('IDRiD_', ''))

test_df_eval = test_df[test_df['Image name'].apply(img_num).isin(frozen_nums)].copy()
test_df_rest = test_df[~test_df['Image name'].apply(img_num).isin(frozen_nums)].copy()

# Use non-frozen test images as part of val set
val_df = pd.concat([test_df_eval, test_df_rest], ignore_index=True)

train_dataset = IDRiDDataset(train_df, TRAIN_IMG, train_transform)
val_dataset   = IDRiDDataset(val_df,   TEST_IMG,  val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

# Class weights (same as week1_training.ipynb)
labels_arr  = train_df['Retinopathy grade'].values
classes     = np.arange(NUM_CLASSES)
cw          = compute_class_weight('balanced', classes=classes, y=labels_arr)
class_weights = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

print(f'Train: {len(train_dataset)} images | Val: {len(val_dataset)} images')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Class weights: {cw.round(3)}')


In [ ]:
# Cell 5: Fine-tune ViT-B16
# Skips training if vit_b16_best.pth already exists on Drive.

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out   = model(imgs)
            loss  = criterion(out, labels)
            preds = out.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return total_loss / total, correct / total, kappa

if os.path.exists(CKPT_PATH):
    print(f'Checkpoint already exists at {CKPT_PATH} — skipping training.')
    print('Delete it from Drive if you want to retrain from scratch.')
else:
    print('Starting ViT-B16 fine-tuning...')
    model = timm.create_model('vit_base_patch16_224', pretrained=True,
                               num_classes=NUM_CLASSES)
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=NUM_EPOCHS)

    best_kappa = -1.0
    history    = {'train_loss': [], 'val_loss': [], 'val_kappa': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc            = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, va_kappa  = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['val_kappa'].append(va_kappa)

        elapsed = time.time() - t0
        print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
              f'Loss {tr_loss:.4f}/{va_loss:.4f} | '
              f'Acc {tr_acc:.3f}/{va_acc:.3f} | '
              f'Kappa {va_kappa:.4f} | {elapsed:.1f}s')

        if va_kappa > best_kappa:
            best_kappa = va_kappa
            torch.save(model.state_dict(), CKPT_PATH)
            print(f'  ✓ Saved best checkpoint (kappa={best_kappa:.4f})')

    print(f'\nBest val kappa: {best_kappa:.4f}')
    print(f'Checkpoint saved to: {CKPT_PATH}')

    # Loss curve
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], label='train')
    axes[0].plot(epochs, history['val_loss'],   label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(epochs, history['val_kappa'], color='darkorange')
    axes[1].set_title('Val Quadratic Kappa')
    plt.tight_layout(); plt.show()

    del model
    torch.cuda.empty_cache()


In [ ]:
# Cell 6: Load checkpoint + generate attention rollout heatmaps

# ── Load model ────────────────────────────────────────────────────────────────
model = timm.create_model('vit_base_patch16_224', pretrained=False,
                           num_classes=NUM_CLASSES)
state_dict = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()
print(f'Loaded checkpoint: {CKPT_PATH}')

# ── Attention rollout hooks ───────────────────────────────────────────────────
# How it works:
#   1. Hook every transformer block's attention module
#   2. Re-compute attention weights (Q@K^T softmax) from the QKV projection
#   3. Average over heads, add 0.5*I residual, multiply sequentially across layers
#   4. Row 0 (CLS token) of the result gives each patch's importance
#   5. Reshape to (14,14) grid, upsample to original image resolution

_attn_maps = []

def _hook(module, input, output):
    B, N, C = input[0].shape
    qkv = module.qkv(input[0]).reshape(
              B, N, 3, module.num_heads, C // module.num_heads).permute(2, 0, 3, 1, 4)
    q, k, _ = qkv.unbind(0)
    attn = (q @ k.transpose(-2, -1)) * module.scale
    attn = attn.softmax(dim=-1)
    _attn_maps.append(attn.detach().cpu())

hooks = [block.attn.register_forward_hook(_hook) for block in model.blocks]
print(f'Hooks registered on {len(hooks)} transformer blocks')

VIT_TRANSFORM = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def load_image(image_id):
    num = int(image_id.replace('IDRiD_', ''))
    return Image.open(os.path.join(TEST_IMG, f'IDRiD_{num:03d}.jpg')).convert('RGB')

def clean_id(image_id):
    return f'IDRiD_{int(image_id.replace("IDRiD_", ""))}'

def attention_rollout(pil_img):
    global _attn_maps
    _attn_maps = []

    tensor = VIT_TRANSFORM(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
    pred_class = logits.argmax(dim=1).item()

    n = _attn_maps[0].shape[-1]   # num_patches + 1 (CLS)
    I = torch.eye(n)
    rollout = I.clone()
    for attn in _attn_maps:
        a = attn[0].mean(dim=0)           # avg over heads
        a = 0.5 * a + 0.5 * I            # add residual
        a = a / a.sum(dim=-1, keepdim=True)
        rollout = a @ rollout

    patch_scores = rollout[0, 1:].numpy()              # (196,)
    grid = int(patch_scores.shape[0] ** 0.5)           # 14
    hm   = patch_scores.reshape(grid, grid)

    orig_w, orig_h = pil_img.size
    hm_t = torch.from_numpy(hm).unsqueeze(0).unsqueeze(0).float()
    hm_t = F.interpolate(hm_t, size=(orig_h, orig_w),
                          mode='bilinear', align_corners=False)
    hm   = hm_t.squeeze().numpy()

    mn, mx = hm.min(), hm.max()
    hm = (hm - mn) / (mx - mn + 1e-8)
    return hm.astype(np.float32), pred_class

# ── Generate heatmaps ─────────────────────────────────────────────────────────
pred_records = []

for i, image_id in enumerate(TEST_IDS):
    cid      = clean_id(image_id)
    npy_path = os.path.join(HEATMAP_DIR, f'vit_b16_attention_rollout_{cid}.npy')

    if os.path.exists(npy_path):
        print(f'  [{i+1:02d}/27] {cid} already done — skipping')
        continue

    try:
        pil_img = load_image(image_id)
        heatmap, pred_class = attention_rollout(pil_img)

        # Verify output contract
        assert heatmap.ndim  == 2
        assert heatmap.dtype == np.float32
        assert heatmap.shape == (pil_img.size[1], pil_img.size[0])
        assert heatmap.min() >= 0.0 and heatmap.max() <= 1.0

        np.save(npy_path, heatmap)

        # Overlay PNG
        orig  = np.array(pil_img.resize((heatmap.shape[1], heatmap.shape[0])))
        color = (cm.jet(heatmap)[:, :, :3] * 255).astype(np.uint8)
        blend = (0.5 * orig + 0.5 * color).astype(np.uint8)
        Image.fromarray(blend).save(
            os.path.join(OVERLAY_DIR, f'{cid}_overlay.png'))

        pred_records.append({'image_id': cid, 'model': 'vit_b16',
                             'method': 'attention_rollout',
                             'predicted_grade': pred_class})

        print(f'  [{i+1:02d}/27] {cid}: shape={heatmap.shape} '
              f'min={heatmap.min():.3f} max={heatmap.max():.3f} pred={pred_class}')

    except Exception as e:
        print(f'  [{i+1:02d}/27] {image_id} FAILED: {e}')

print(f'\nDone. {len(pred_records)} heatmaps saved to {HEATMAP_DIR}')


In [ ]:
# Cell 7: Verification + visual sanity check
import glob

npy_files = glob.glob(os.path.join(HEATMAP_DIR, '*.npy'))
png_files = glob.glob(os.path.join(OVERLAY_DIR, '*_overlay.png'))

print(f'Heatmaps: {len(npy_files)}/27')
print(f'Overlays: {len(png_files)}/27')

# Visual check — first image
if npy_files:
    sample_id = clean_id(TEST_IDS[0])
    pil = load_image(TEST_IDS[0])
    hm  = np.load(os.path.join(HEATMAP_DIR,
                  f'vit_b16_attention_rollout_{sample_id}.npy'))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(pil); axes[0].set_title('Original'); axes[0].axis('off')
    orig  = np.array(pil.resize((hm.shape[1], hm.shape[0])))
    color = (cm.jet(hm)[:, :, :3] * 255).astype(np.uint8)
    blend = (0.5 * orig + 0.5 * color).astype(np.uint8)
    axes[1].imshow(blend)
    axes[1].set_title(f'Attention Rollout — {sample_id}')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

status = 'COMPLETE' if len(npy_files) == 27 else f'INCOMPLETE ({27 - len(npy_files)} missing)'
print(f'\nViT heatmaps {status} — {"ready for aopc.ipynb" if len(npy_files) == 27 else "re-run Cell 6"}')
